# Answers: 1 · Meet the data

Worked answers to the **Try it yourself** exercises at the end of
[`1_meet_the_data.ipynb`](../1_meet_the_data.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

In [ ]:
import sys; sys.path.insert(0, "..")            # tour.py lives one folder up
import numpy as np
from rdkit import Chem
from tour import *

scores, embeddings = load_mini()

def similarity(table, i):
    unit = table / np.linalg.norm(table, axis=1, keepdims=True)
    return unit @ unit[i]

def n_atoms(smi):
    return Chem.MolFromSmiles(smi).GetNumHeavyAtoms()
print("ready")

---
## Exercise 1: the molecules closest to the **worst** one. Are they small?

In [ ]:
worst = scores["score"].idxmax()                    # idxmax: the row with the LARGEST (= worst) score
sim = similarity(embeddings["unimol2"], worst)
closest = np.argsort(sim)[::-1][:8]                 # includes the worst molecule itself, first
draw(scores["smiles"][closest], labels=[f"{scores['score'][i]}  ({n_atoms(scores['smiles'][i])} atoms)" for i in closest])

In [ ]:
neighbour_atoms = [n_atoms(scores["smiles"][i]) for i in closest[1:]]
all_atoms = [n_atoms(s) for s in scores["smiles"]]
print("heavy atoms in the 7 neighbours:", neighbour_atoms)
print(f"average size: neighbours {np.mean(neighbour_atoms):.1f} atoms, all molecules {np.mean(all_atoms):.1f} atoms")
print(f"average score: neighbours {scores['score'][closest[1:]].mean():.2f}, all molecules {scores['score'].mean():.2f}")

**Answer.** Yes. The worst molecule's neighbours are small: about 11 heavy atoms on average, against 24.5 for the whole pool. They also score badly (−6.3 on average, against −7.9 overall). That fits notebook 1's finding that docking favours bigger molecules. Small molecules have little surface to touch the protein, so they rarely score well, and Uni-Mol2 places them near each other because it reads 3D shape, and small shapes resemble each other.

---
## Exercise 2: nearest neighbours with MoLFormer instead of Uni-Mol2

In [ ]:
best = scores["score"].idxmin()
for name in ["unimol2", "molformer"]:
    sim = similarity(embeddings[name], best)
    closest = np.argsort(sim)[::-1][:5]
    print(f"{name:10s} neighbours' scores: {list(scores['score'][closest[1:]])}   similarities: {np.round(sim[closest[1:]], 3).tolist()}")

In [ ]:
sim = similarity(embeddings["molformer"], best)
closest = np.argsort(sim)[::-1][:5]
draw(scores["smiles"][closest], labels=[f"{scores['score'][i]}  (sim {sim[i]:.2f})" for i in closest], per_row=5)

**Answer.** The neighbours MoLFormer picks also score well here (−8.0 to −9.6), about as well as Uni-Mol2's. For this one molecule the two look similar. Over the 30 best molecules in the lesson, though, MoLFormer's neighbours averaged −8.75 against Uni-Mol2's −9.14, so on average its idea of "similar" matches docking less well.

Look at the similarity numbers too, because they're a trap. Uni-Mol2 says every neighbour has similarity **0.999**. That doesn't mean they're nearly identical. In Uni-Mol2's space even two *random* molecules have a similarity of about 0.997 (median), because its 768 numbers mostly move together, so all its vectors point in almost the same direction. The project documents exactly this (`al-molecular/docs/adaptive_acq_results.md`, §8.5: the ENHITS Uni-Mol2 embedding is "degenerate"). For comparison, random pairs score about 0.83 in MoLFormer and 0.34 in MHG-GED. **Only the order of the similarities means something, not their size.** That's why the nearest-neighbour test still works for Uni-Mol2, and why it's still the most useful backbone in notebook 3.

---
## Exercise 3: what fraction of molecules score better than −10?

In [ ]:
better = scores["score"] < -10                # True/False for every molecule
print("fraction:", round(better.mean(), 4), "  count:", better.sum(), "of", len(scores))

**Answer.** 0.92%: 193 of the 21,044 molecules score better than −10. The best 100, the tour's target, are about half of those. So "better than −10" is already a rare, excellent molecule, and finding these few in a sea of average ones is the whole challenge.